# quanteo - Asian Option Pricing
**Author:** Matteo Ientile | **Environment:** Python 3.10+ (`quanteo` library)

This notebook demonstrates the pricing of Arithmetic and Geometric Asian Options

In [2]:
import numpy as np

# Quanteo Libraries
from quanteo.options import EuropeanOption, SumPricesCV, ArithmeticAsianOption, GeometricAsianOption
from quanteo.pricers import MonteCarloPricer, QuasiMCPricer, BSMPricer, ControlVariateMC, GeometricAsianPricer
from quanteo.models import GBM
from quanteo.risk import AnalyticalBSMGreeks, FiniteDifferenceGreek

In [1]:
# 1. ==================== INPUT DATA & SIMULATION CONSTRAINTS

S0 = 85.0 # current asset price
K = 88.0 # strike price
T = 0.5 # years, i.e. 6 months
t_time = 0 # -> T = time to maturity
r = 0.04 #risk free rate
sigma = 0.42 # volatility
N = 21*6 # consider daily prices for the average --> 21 trading days/month * 6 months
option_type="call" #call option

n_paths = 5_000 # number of trajectories
SEED = 42 # reproducibility
alpha = 0.95 # confidence interval level
antithetic = False # do not overlap different variance reduction strategies

In [3]:
# 2. ==================== DEFINE CONTRACTS & HOW TO MODEL UNDERLYING ASSET PRICE
#Arithmetic Asian Option to price
option = ArithmeticAsianOption(
    T=T,
    K=K,
    N=N,
    option_type=option_type
)

# Analogue option, but considering Geometric mean rather than arithmetic
option_geom = GeometricAsianOption(
    T=T,
    K=K,
    N=N,
    option_type=option_type
)

#How to model the underlying asset 
model = GBM(
    S0=S0,
    r=r,
    sigma=sigma,
    t_time=0.0
)

In [4]:
# 3. ==================== DEFINE PRICERS
# Plain MC
crude_mc_pricer = MonteCarloPricer(
    n_paths=n_paths,
    n_steps=N, 
    seed=SEED,
    alpha=alpha,
    antithetic=antithetic
)
# Exact solution with Geometric Averge
geometric_pricer = GeometricAsianPricer()

# Control Variate pricer
cv_mc_pricer = ControlVariateMC(
    mc_pricer=crude_mc_pricer 
)

In [5]:
# 4. ==================== COMPUTE PAYOFFS

# - Crude Monte Carlo
crude_mc_result = crude_mc_pricer.price(
    option=option,
    model=model
)
crude_mc_price = crude_mc_result.price #payoff according to Plain Monte Carlo, no variance reduction applied
crude_mc_ci = crude_mc_result.metrics["confidence interval"] #confidence intervals associated to Plain Monte Carlo
crude_mc_ciw = crude_mc_ci[1] - crude_mc_ci[0] #confidence interval width associated to Plain Monte Carlo

# - Geometric Option ANALYTICAL Solution
geometric_exact_price = geometric_pricer.price(
    option=option_geom,
    model=model
).price

# - Monte Carlo with Geometric Payoff as Control Variate
cv_geo_mc_results = cv_mc_pricer.price(
    option=option, 
    control_var=option_geom,
    exact_cv_price=geometric_exact_price,
    model=model,
    n_pilot=1_000 
)
cv_geo_mc_price = cv_geo_mc_results.price 
cv_geo_mc_ci = cv_geo_mc_results.metrics["confidence_interval"]
cv_geo_mc_ciw = cv_geo_mc_ci[1] - cv_geo_mc_ci[0] #confidence interval width associated to MC with VR (Sum as control variate)
cv_geo_mc_cstar = cv_geo_mc_results.metrics["c_star"]


In [7]:
# 5. ==================== PRINT RESULTS

title = "QUANT LIBRARY TEST - ASIAN CALL OTM "
subtitle_contract = "--- CONTRACT CHARACTERISTICS ---"
subtitle_sim = "--- SIMULATION PARAMETERS ---"
subtitle_price = "--- PRICING ---"
subtitle_variance = "--- VARIANCE REDUCTION GAIN ---"

#title
print("="*100)
print(f"{title:^100}")
print("="*100)
print("")

#contract section
print(f"{subtitle_contract:^100}")
print("")
print(f"{'Current Asset Price':<25}: {S0}")
print(f"{'Strike Price':<25}: {K}")
print(f"{'Time-to-Maturity':<25}: {T}")
print(f"{'Risk-free-rate':<25}: {r}")
print(f"{'Volatility':<25}: {sigma}")
print(f"{'Observation Frequency':<25}: {N}, daily prices")
print("")

#simulation section
print(f"{subtitle_sim:^100}")
print("")
print(f"{'Number of simulations':<25}: {n_paths}")
print(f"{'Confidence Intervals level':<25}: {alpha*100}%")
print("")

print("="*100)
#prices section
print(f"{subtitle_price:^100}")
print("")
print(f"{'Crude MC Price':<50} : {crude_mc_price:.4f} | {alpha*100}% CI : [{crude_mc_ci[0]:.4f}, {crude_mc_ci[1]:.4f}], width = {crude_mc_ciw:.4f}")
print(f"{'Exact (Geometric Avg) Price':<50} : {geometric_exact_price:.4f}")
print(f"{'Control Variate MC Price':<50} : {cv_geo_mc_price:.4f} | {alpha*100}% CI : [{cv_geo_mc_ci[0]:.4f}, {cv_geo_mc_ci[1]:.4f}], width = {cv_geo_mc_ciw:.4f}, c* = {cv_geo_mc_cstar:.4f}")
print("")

#variance reduction section
print(f"{subtitle_variance:^100}")
print(f"{'Width of Confidence Interval reduced by':<50} : {(1 - (cv_geo_mc_ciw / crude_mc_ciw)) * 100:.4f} %")


                                QUANT LIBRARY TEST - ASIAN CALL OTM                                 

                                  --- CONTRACT CHARACTERISTICS ---                                  

Current Asset Price      : 85.0
Strike Price             : 88.0
Time-to-Maturity         : 0.5
Risk-free-rate           : 0.04
Volatility               : 0.42
Observation Frequency    : 126, daily prices

                                   --- SIMULATION PARAMETERS ---                                    

Number of simulations    : 5000
Confidence Intervals level: 95.0%

                                          --- PRICING ---                                           

Crude MC Price                                     : 4.7275 | 95.0% CI : [4.4818, 4.9732], width = 0.4914
Exact (Geometric Avg) Price                        : 4.5781
Control Variate MC Price                           : 4.8862 | 95.0% CI : [4.8758, 4.8965], width = 0.0207, c* = 1.0596

                                  